In [ ]:
import json
import urllib.error
import urllib.request
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import scipy.stats as stats
import itertools

import altair as alt
from sklearn.metrics import precision_recall_curve, auc

# Overhead for finding data

In [ ]:
input_directory = Path('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/sge_data_for_qc')
scores_excel = Path('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/20260101_SGEsubset.xlsx')

all_scores = pd.read_excel(scores_excel)

In [ ]:
def find_genes(input_dir: Path | None = None) -> dict:
    """Discover all gene datasets in the input directory.

    Detects genes by finding all *delcounts.tsv files and extracting the gene
    name from each filename. Both dot-separated (e.g. CTCF.delcounts.tsv) and
    run-together (e.g. 20260129_RAD51Ddelcounts.tsv) naming conventions are
    supported. Companion files (*snvcounts.tsv, *editrates.tsv) must also be
    present for each detected gene. Per-gene scores are sourced from a shared
    Excel file (filtered by the 'Gene' column) passed via scores_excel.

    Args:
        input_dir: Directory containing gene-specific TSV files.
        scores_excel: Path to the shared scores Excel file (e.g.
            20260101_SGEsubset.xlsx). Scores for each gene are obtained by
            filtering rows where Gene == gene_name.

    Returns a dict mapping gene name -> files dict, e.g.:
        {"RAD51D": {"del_counts": Path(...), "snv_counts": Path(...), ...}}
    """
    delcounts_files = sorted(input_dir.glob("*delcounts.tsv"))
    if not delcounts_files:
        raise FileNotFoundError(f"No '*delcounts.tsv' files found in {input_dir}")

    def find_one(*patterns):
        for pattern in patterns:
            matches = list(input_dir.glob(pattern))
            if len(matches) == 1:
                return matches[0]
            if len(matches) > 1:
                raise ValueError(
                    f"Multiple files match '{pattern}': "
                    + ", ".join(str(m) for m in matches)
                )
        raise FileNotFoundError(
            f"Could not find any of {patterns} in {input_dir}"
        )

    genes = {}
    for delcounts_path in delcounts_files:
        # Handles both GENE.delcounts.tsv and GENEdelcounts.tsv (with optional prefix)
        stem_part = delcounts_path.stem.split("_")[-1]
        gene = stem_part.removesuffix(".delcounts").removesuffix("delcounts")

        gene_score_df = all_scores.loc[all_scores['Gene'] == gene].copy()
        genes[gene] = {
            "del_counts": delcounts_path,
            "snv_counts": find_one(f"*{gene}snvcounts.tsv", f"*{gene}.snvcounts.tsv"),
            "edit_rates": find_one(f"*{gene}editrates.tsv", f"*{gene}.editrates.tsv"),
            "scores_excel": gene_score_df
        }

    return genes

In [ ]:
gene_paths= find_genes(input_directory)

# Helper functions to build visualizations

In [ ]:
#Read all data into dictionary of dataframes
def read_data(gene):
    data_dict = {}
    paths = gene_paths[gene]

    to_read = ['del_counts', 'snv_counts', 'edit_rates']

    for elem in to_read:
        df = pd.read_csv(paths[elem], sep = '\t')
        data_dict[elem] = df
    
    data_dict['scores'] = paths['scores_excel']
    
    return data_dict

# Builds standard QC viz. for each gene

In [ ]:
for gene in gene_paths:
    data_dict = read_data(gene)
    print(data_dict)